GT4Py - GridTools Framework

Copyright (c) 2014-2024, ETH Zurich
All rights reserved.

Please, refer to the LICENSE file in the root directory.
SPDX-License-Identifier: BSD-3-Clause

# Shallow Water Model on the TPU

Companion to `swm.ipynb`: the same doubly periodic shallow water model on an Arakawa
C-grid, leapfrog in time with a Robert-Asselin filter, but with the eight
stencil operators rewritten so the stencil work lands on the TPU's MXU
(matrix multiply unit) instead of the VPU (vector unit).

The TPU's TensorCore splits its work between the **VPU**, which runs elementwise
arithmetic and is limited by HBM bandwidth, and the **MXU**, a systolic array
providing the bulk of the chip's floating-point throughput. From JAX, only
`dot`-family contractions and `conv_general_dilated` engage the MXU: XLA lowers
a convolution to a GEMM over im2col windows, reading each input element once
into vector memory (VMEM) and contracting on the sliding window. A stencil
expressed as explicit shifts, as in `swm.ipynb`, produces slices and elementwise
chains: pure VPU work, with the MXU idle. Everything is also float32 here,
because the MXU has no float64.

The restructure splits the timestep into two *linear stages* and two *pointwise
stages*, because the scheme nests linear operators around its non-linearities
(products and a division):

```
GEMM 1 (conv) -> kernel_terms (field operator) -> GEMM 2 (conv) -> leapfrog (field operator)
```

Each GEMM is one batched convolution over a channel stack that computes *every*
shifted quantity the scheme needs as an output channel; the two field operators
then combine those channels with the prognostic fields, pointwise. Read
`swm.ipynb` for the model itself; only what is different is explained here.

Three differences from `swm.ipynb`:

- **No halo.** The fields hold only the interior; periodicity is a circular
padding inside the convolutions, so there is no host-side halo refresh between
steps at all.
- **float32 instead of float64**, for the MXU. The validation section measures
what that costs.
- **Self-contained.** The NumPy reference and the NCAR benchmark data below are
inlined, so the notebook runs on its own -- from a Colab TPU runtime, for
example, without a checkout of the repository.

## The NumPy reference

The content of `swm_numpy.py`, the NumPy implementation of the same scheme that
`swm.ipynb` validates against, inlined so that this notebook runs on its own
as well. Its constants and functions are used as-is below: `initial_conditions`
to set up the fields, `timestep` to time-step the reference, and `run` for the
whole integration. Its `run` is not to be confused with this notebook's
`run_model`, which integrates the MXU formulation through a GT4Py backend.

In [ ]:
# The content of `swm_numpy.py`, inlined verbatim below.

"""
NumPy reference for the shallow water model example.

Companion to `swm.ipynb`. The model is doubly periodic, so periodicity is expressed here
with `numpy.roll` on interior-only arrays of shape `(M, N)` -- there is no halo. The
notebook's GT4Py version instead carries one halo cell per side and updates it explicitly,
which is what makes the two implementations worth comparing.
"""

from __future__ import annotations

import numpy as np


# Model constants of the NCAR/SWM benchmark configuration.
M = 16
N = 16
DX = 100000.0
DY = 100000.0
DT = 90.0
ALPHA = 0.001
A = 1000000.0
P_MEAN = 50000.0
ITMAX = 4000


def avg_x(f: np.ndarray) -> np.ndarray:
    return 0.5 * (np.roll(f, -1, axis=0) + f)


def avg_y(f: np.ndarray) -> np.ndarray:
    return 0.5 * (np.roll(f, -1, axis=1) + f)


def avg_x_staggered(f: np.ndarray) -> np.ndarray:
    return 0.5 * (np.roll(f, 1, axis=0) + f)


def avg_y_staggered(f: np.ndarray) -> np.ndarray:
    return 0.5 * (np.roll(f, 1, axis=1) + f)


def delta_x(dx: float, f: np.ndarray) -> np.ndarray:
    return (np.roll(f, -1, axis=0) - f) / dx


def delta_y(dy: float, f: np.ndarray) -> np.ndarray:
    return (np.roll(f, -1, axis=1) - f) / dy


def delta_x_staggered(dx: float, f: np.ndarray) -> np.ndarray:
    return (f - np.roll(f, 1, axis=0)) / dx


def delta_y_staggered(dy: float, f: np.ndarray) -> np.ndarray:
    return (f - np.roll(f, 1, axis=1)) / dy


def initial_conditions(
    m: int = M, n: int = N, dx: float = DX, dy: float = DY, a: float = A, p_mean: float = P_MEAN
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Return the initial `u`, `v`, `p` on the interior grid, each of shape `(m, n)`.

    The velocity field is derived from a doubly periodic stream function
    `psi = a * sin(2*pi*(i+0.5)/m) * sin(2*pi*(j+0.5)/n)`.
    """
    d_i = 2.0 * np.pi / m
    d_j = 2.0 * np.pi / n
    el = n * dx
    pcf = (np.pi * np.pi * a * a) / (el * el)

    psi = (
        a
        * np.sin((np.arange(0, m + 1).reshape(-1, 1) + 0.5) * d_i)
        * np.sin((np.arange(0, n + 1) + 0.5) * d_j)
    )
    p = (
        pcf
        * (np.cos(2.0 * np.arange(0, m).reshape(-1, 1) * d_i) + np.cos(2.0 * np.arange(0, n) * d_j))
        + p_mean
    )
    u = -(psi[1:, 1:] - psi[1:, :-1]) / dy
    v = (psi[1:, 1:] - psi[:-1, 1:]) / dx

    return u, v, p


def timestep(
    u: np.ndarray,
    v: np.ndarray,
    p: np.ndarray,
    uold: np.ndarray,
    vold: np.ndarray,
    pold: np.ndarray,
    dx: float,
    dy: float,
    tdt: float,
    alpha: float,
) -> tuple[np.ndarray, ...]:
    """One leapfrog step with a Robert-Asselin time filter. Returns the six updated fields."""
    cu = avg_x(p) * u
    cv = avg_y(p) * v
    z = (delta_x(dx, v) - delta_y(dy, u)) / avg_x(avg_y(p))
    h = p + 0.5 * (avg_x_staggered(u * u) + avg_y_staggered(v * v))

    unew = uold + avg_y_staggered(z) * avg_y_staggered(avg_x(cv)) * tdt - delta_x(dx, h) * tdt
    vnew = vold - avg_x_staggered(z) * avg_x_staggered(avg_y(cu)) * tdt - delta_y(dy, h) * tdt
    pnew = pold - delta_x_staggered(dx, cu) * tdt - delta_y_staggered(dy, cv) * tdt

    uold_new = u + alpha * (unew - 2.0 * u + uold)
    vold_new = v + alpha * (vnew - 2.0 * v + vold)
    pold_new = p + alpha * (pnew - 2.0 * p + pold)

    return unew, vnew, pnew, uold_new, vold_new, pold_new


def run(
    itmax: int,
    m: int = M,
    n: int = N,
    dx: float = DX,
    dy: float = DY,
    dt: float = DT,
    alpha: float = ALPHA,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Integrate `itmax` steps and return the final interior `u`, `v`, `p`."""
    u, v, p = initial_conditions(m, n, dx, dy)
    uold, vold, pold = u.copy(), v.copy(), p.copy()

    for cycle in range(itmax):
        # The first step is forward Euler and unfiltered; afterwards leapfrog with tdt = 2*dt.
        tdt = dt if cycle == 0 else 2.0 * dt
        step_alpha = 0.0 if cycle == 0 else alpha
        unew, vnew, pnew, uold, vold, pold = timestep(
            u, v, p, uold, vold, pold, dx, dy, tdt, step_alpha
        )
        u, v, p = unew, vnew, pnew

    return u, v, p


def to_reference_layout(
    u: np.ndarray, v: np.ndarray, p: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Convert interior `(M, N)` fields to the `(M+1, N+1)` layout of the NCAR/SWM dumps.

    The extra row and column are the periodic images the original Fortran carried
    explicitly; which side they sit on differs per field because of the C-grid staggering.
    """
    return (
        np.pad(u, ((1, 0), (0, 1)), mode="wrap"),
        np.pad(v, ((0, 1), (1, 0)), mode="wrap"),
        np.pad(p, ((0, 1), (0, 1)), mode="wrap"),
    )

## Grid, field type and backend

`gtx_experimental.jax_jit` compiles the two field operators with `jax.jit` and
calls the compiled functions from then on. The convolutions are jitted
separately, so one step dispatches four compiled kernels back to back, without
host work in between.

The default device of JAX is used everywhere. On a machine with a TPU runtime
(a Colab kernel with a TPU accelerator, for example), `jax.devices()` reports
the TPU and every array lands on it; GT4Py needs no explicit device for this.

In [ ]:
from typing import Any

import numpy as np
import jax
import jax.numpy as jnp

import gt4py.next as gtx
from gt4py.next import astype, experimental as gtx_experimental, float32
from gt4py._core import definitions as core_defs

backend = gtx_experimental.jax_jit
# backend = gtx_experimental.embedded_jax  # eager execution, no jit

print("backend:", backend.name)
print("devices:", jax.devices())

The grid is the same as in `swm.ipynb`, but the fields span only the interior:
without a halo there is no `apply_periodic`, and the model fields carry just
`m * n` values.

All scalars that change during the integration (`tdt`, `alpha`) are traced
arguments of the `leapfrog` operator, so the whole 4000-step benchmark reuses
one compiled function per grid size.

In [ ]:
I = gtx.Dimension("I")
J = gtx.Dimension("J")

IJField = gtx.Field[gtx.Dims[I, J], gtx.float32]

F32 = jnp.float32

# the model constants M, N, DX, DY, DT, ALPHA come from the reference above

## The linear stages: shifts as a GEMM

Both linear stages are one `conv_general_dilated` over a channel stack in
`NHWC` layout (channels minor, which is what XLA wants), filters of shape
`[3, 3, C_in, C_out]`. The stack is circularly padded by one cell on every side
and convolved with `VALID` padding, so periodicity is interior to the kernel.

A tap `(a, b)` of the filter, `a` and `b` from `{0, 1, 2}`, multiplies the
input at `(i + a - 1, j + b - 1)` of output point `(i, j)`. In these units the
eight operators of `swm.ipynb` are two-tap filters, and the whole first linear
stage is:

| channel | formula | taps |
| --- | --- | --- |
| `avgx_p` | $\overline{P}^{\,x}$ = ½(P[i+1] + P[i]) | `(2,1)`, `(1,1)` |
| `avgy_p` | $\overline{P}^{\,y}$ | `(1,2)`, `(1,1)` |
| `avgxy_p` | $\overline{\overline{P}}^{\,xy}$ = ¼ P[window i..i+1, j..j+1] | the four `(a,b)` in `{1,2}×{1,2}` |
| `dx_v` | δ_x V = (V[i+1] − V[i])/dx | `(2,1)` +1/dx, `(1,1)` −1/dx |
| `dy_u` | δ_y U | `(1,2)` +1/dy, `(1,1)` −1/dy |
| `avgxs_u2` | backward ½(U²[i−1] + U²[i]) | `(0,1)`, `(1,1)` |
| `avgys_v2` | backward average of V² in y | `(1,0)`, `(1,1)` |

The `leapfrog` operator of `swm.ipynb` then combines these with its
non-linearities (`kernel_terms` below), and the second linear stage is:

| channel | formula | taps |
| --- | --- | --- |
| `avgys_z` | backward average of Z in y | `(1,0)`, `(1,1)` |
| `avgxs_z` | backward average of Z in x | `(0,1)`, `(1,1)` |
| `avgys_avgx_cv` | $\overline{\overline{V\overline{P}^{\,y}}^{\,x}}^{\,y}$ | `{1,2}×{0,1}` at ¼ |
| `avgxs_avgy_cu` | $\overline{\overline{U\overline{P}^{\,x}}^{\,y}}^{\,x}$ | `{0,1}×{1,2}` at ¼ |
| `dx_h` | δ_x H | `(2,1)` ±1/dx |
| `dy_h` | δ_y H | `(1,2)` ±1/dy |
| `dxs_cu` | backward δ_x of $U\overline{P}^{\,x}$ | `(1,1)` +1/dx, `(0,1)` −1/dx |
| `dys_cv` | backward δ_y of $V\overline{P}^{\,y}$ | `(1,1)` +1/dy, `(1,0)` −1/dy |

Each convolution lowers to one implicit-im2col GEMM of `M = m·n` spatial rows,
`K = 9·C_in` contraction (the whole window), and `N = C_out` — which is why
every shifted quantity of the stage is packed into a single call: the GEMM
pads `K` and `N` up to multiples of the systolic array size (128 on pre-v6e
TPUs, 256 on v6e and 7x), and the more of that padded width carries real
stencil work, the better the MXU is used.

In [ ]:
IN1 = ["u", "v", "p", "u squares", "v squares"]
OUT1 = ["avgx_p", "avgy_p", "avgxy_p", "dx_v", "dy_u", "avgxs_u2", "avgys_v2"]

IN2 = ["z", "cu", "cv", "h"]
OUT2 = [
    "avgys_z",
    "avgxs_z",
    "avgys_avgx_cv",
    "avgxs_avgy_cu",
    "dx_h",
    "dy_h",
    "dxs_cu",
    "dys_cv",
]


def build_kernel(
    inputs: list[str],
    outputs: list[str],
    specs: dict[tuple[str, str], list[tuple[int, int, float]]],
) -> jax.Array:
    """
    Build one `[3, 3, C_in, C_out]` filter from tap specifications.

    Args:
        inputs: the names of the input channels, in stack order.
        outputs: the names of the output channels, in stack order.
        specs: maps each `(output channel, input channel)` pair to its list of
            `(a, b, coefficient)` taps: tap `(a, b)` of output point `(i, j)`
            multiplies the input at `(i + a - 1, j + b - 1)`.

    Returns:
        the filter, in `HWIO` layout.
    """
    kernel = jnp.zeros((3, 3, len(inputs), len(outputs)), dtype=F32)
    for (out, inp), taps in specs.items():
        for a, b, coefficient in taps:
            kernel = kernel.at[a, b, inputs.index(inp), outputs.index(out)].set(F32(coefficient))
    return kernel


K1 = build_kernel(
    IN1,
    OUT1,
    {
        ("avgx_p", "p"): [(1, 1, 0.5), (2, 1, 0.5)],
        ("avgy_p", "p"): [(1, 1, 0.5), (1, 2, 0.5)],
        ("avgxy_p", "p"): [(a, b, 0.25) for a in (1, 2) for b in (1, 2)],
        ("dx_v", "v"): [(1, 1, -1.0 / DX), (2, 1, 1.0 / DX)],
        ("dy_u", "u"): [(1, 1, -1.0 / DY), (1, 2, 1.0 / DY)],
        ("avgxs_u2", "u squares"): [(0, 1, 0.5), (1, 1, 0.5)],
        ("avgys_v2", "v squares"): [(1, 0, 0.5), (1, 1, 0.5)],
    },
)

K2 = build_kernel(
    IN2,
    OUT2,
    {
        ("avgys_z", "z"): [(1, 0, 0.5), (1, 1, 0.5)],
        ("avgxs_z", "z"): [(0, 1, 0.5), (1, 1, 0.5)],
        ("avgys_avgx_cv", "cv"): [(a, b, 0.25) for a in (1, 2) for b in (0, 1)],
        ("avgxs_avgy_cu", "cu"): [(a, b, 0.25) for a in (0, 1) for b in (1, 2)],
        ("dx_h", "h"): [(1, 1, -1.0 / DX), (2, 1, 1.0 / DX)],
        ("dy_h", "h"): [(1, 1, -1.0 / DY), (1, 2, 1.0 / DY)],
        ("dxs_cu", "cu"): [(0, 1, -1.0 / DX), (1, 1, 1.0 / DX)],
        ("dys_cv", "cv"): [(1, 0, -1.0 / DY), (1, 1, 1.0 / DY)],
    },
)

The two stages. `dx` and `dy` are baked into the filters, which are constant
per grid configuration; `tdt` and `alpha` are not part of either stage.

In [ ]:
CONV_SPEC = ("NHWC", "HWIO", "NHWC")  # activations NHWC, filters HWIO, results NHWC


def linear_stage(channels: list[jax.Array], kernel: jax.Array) -> tuple[jax.Array, ...]:
    """One implicit-im2col GEMM over a channel stack; returns the output channels."""
    stack = jnp.stack([jnp.pad(c, ((1, 1), (1, 1)), mode="wrap") for c in channels], axis=-1)
    result = jax.lax.conv_general_dilated(
        stack[None],
        kernel,
        (1, 1),
        ((0, 0), (0, 0)),
        dimension_numbers=CONV_SPEC,
        precision=jax.lax.Precision.DEFAULT,
    )[0]
    return tuple(result[..., i] for i in range(result.shape[-1]))


@jax.jit
def linear_stage_1(u: jax.Array, v: jax.Array, p: jax.Array) -> tuple[jax.Array, ...]:
    """The seven shifted quantities the first pointwise stage consumes."""
    return linear_stage([u, v, p, u * u, v * v], K1)


@jax.jit
def linear_stage_2(
    z: jax.Array, cu: jax.Array, cv: jax.Array, h: jax.Array
) -> tuple[jax.Array, ...]:
    """The eight shifted quantities the leapfrog operator consumes."""
    return linear_stage([z, cu, cv, h], K2)

## The pointwise stages as field operators

With all shifts precomputed as channels, the two remaining stages are purely
pointwise and stay `gt4py.next` field operators. They evaluate the same
equations as the `timestep` operator of `swm.ipynb`, quoted here with the
channel names:

$$
cu = \overline{P}^{\,x} u, \qquad cv = \overline{P}^{\,y} v, \qquad
Z = \frac{\delta_x V - \delta_y U}{\overline{\overline{P}}^{\,xy}}, \qquad
H = P + \tfrac{1}{2}\left(\overline{U^2}^{\,x} + \overline{V^2}^{\,y}\right).
$$

Float literals are float64 in the DSL, and the fields are float32, so the
single literal below converts explicitly; the Robert-Asselin filter of the
`leapfrog` operator is written as `alpha * ((new - cur) + (old - cur))` to the
same effect.

In [ ]:
@gtx.field_operator
def kernel_terms(
    avgx_p: IJField,
    avgy_p: IJField,
    avgxy_p: IJField,
    dx_v: IJField,
    dy_u: IJField,
    avgxs_u2: IJField,
    avgys_v2: IJField,
    u: IJField,
    v: IJField,
    p: IJField,
) -> tuple[IJField, IJField, IJField, IJField]:
    """The scheme's kernel terms, from the first stage's channels."""
    cu = avgx_p * u
    cv = avgy_p * v
    z = (dx_v - dy_u) / avgxy_p
    h = p + astype(0.5, float32) * (avgxs_u2 + avgys_v2)
    return cu, cv, z, h


@gtx.field_operator
def leapfrog(
    avgys_z: IJField,
    avgxs_z: IJField,
    avgys_avgx_cv: IJField,
    avgxs_avgy_cu: IJField,
    dx_h: IJField,
    dy_h: IJField,
    dxs_cu: IJField,
    dys_cv: IJField,
    u: IJField,
    v: IJField,
    p: IJField,
    uold: IJField,
    vold: IJField,
    pold: IJField,
    tdt: gtx.float32,
    alpha: gtx.float32,
) -> tuple[IJField, IJField, IJField, IJField, IJField, IJField]:
    """Advance one leapfrog step. Returns (unew, vnew, pnew, uold, vold, pold)."""
    unew = uold + avgys_z * avgys_avgx_cv * tdt - dx_h * tdt
    vnew = vold - avgxs_z * avgxs_avgy_cu * tdt - dy_h * tdt
    pnew = pold - dxs_cu * tdt - dys_cv * tdt

    # the Robert-Asselin filter, as alpha * ((new - cur) + (old - cur))
    uold_new = u + alpha * ((unew - u) + (uold - u))
    vold_new = v + alpha * ((vnew - v) + (vold - v))
    pold_new = p + alpha * ((pnew - p) + (pold - p))

    return unew, vnew, pnew, uold_new, vold_new, pold_new

## The model

One step dispatches the two jitted convolutions and the two jitted field
operators back to back:

1. GEMM 1 on `u`, `v`, `p`,
2. `kernel_terms` writes `cu`, `cv`, `z`, `h`,
3. GEMM 2 on those,
4. `leapfrog` writes the six updated fields.

JAX arrays are immutable, so an `out=` field of a jitted operator gets a new
buffer rather than being written into; the model reads `.ndarray` off its
fields afterwards, which passes those buffers to the next kernel without a
copy. As in `swm.ipynb`, the first cycle is a forward Euler step with the
filter off, and `tdt` and `alpha` are traced scalars, so scalars changing
after the first step do not recompile anything.

In [ ]:
class Model:
    """The fields of one integration, with its bound operators."""

    def __init__(
        self,
        m: int = M,
        n: int = N,
        *,
        backend: Any,
        device: core_defs.Device | None = None,
    ) -> None:
        self.m, self.n = m, n
        self.domain = gtx.domain({I: (0, m), J: (0, n)})  # interior only, no halo
        self.terms = kernel_terms.with_backend(backend)
        self.update = leapfrog.with_backend(backend)
        self.cycle = 0

        def state(array: np.ndarray | jax.Array) -> gtx.Field:
            return gtx.as_field(
                self.domain, jnp.asarray(array, dtype=F32), allocator=backend, device=device
            )

        def channel(array: jax.Array) -> gtx.Field:
            return gtx.as_field(self.domain, array, allocator=backend)

        u0, v0, p0 = initial_conditions(m, n, DX, DY)
        self.u, self.v, self.p = state(u0), state(v0), state(p0)
        self.uold, self.vold, self.pold = state(u0), state(v0), state(p0)
        # buffers for the stage in between and for the results
        self.unew, self.vnew, self.pnew = state(u0), state(v0), state(p0)
        self.cu, self.cv, self.z, self.h = (state(u0) for _ in range(4))
        self._channel = channel

    def advance(self, cycles: int) -> jax.Array:
        """Integrate `cycles` more steps."""
        for _ in range(cycles):
            tdt = DT if self.cycle == 0 else 2.0 * DT
            alpha = 0.0 if self.cycle == 0 else ALPHA

            stage1 = linear_stage_1(self.u.ndarray, self.v.ndarray, self.p.ndarray)
            self.terms(
                *(self._channel(a) for a in stage1),
                self.u,
                self.v,
                self.p,
                out=(self.cu, self.cv, self.z, self.h),
                domain=self.domain,
                offset_provider={},
            )

            stage2 = linear_stage_2(
                self.z.ndarray, self.cu.ndarray, self.cv.ndarray, self.h.ndarray
            )
            self.update(
                *(self._channel(a) for a in stage2),
                self.u,
                self.v,
                self.p,
                self.uold,
                self.vold,
                self.pold,
                tdt,
                alpha,
                out=(self.unew, self.vnew, self.pnew, self.uold, self.vold, self.pold),
                domain=self.domain,
                offset_provider={},
            )

            self.u, self.unew = self.unew, self.u
            self.v, self.vnew = self.vnew, self.v
            self.p, self.pnew = self.pnew, self.p
            self.cycle += 1

        return self.p.ndarray  # a device buffer, to wait on from the outside

    def arrays(self) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        return (self.u.asnumpy(), self.v.asnumpy(), self.p.asnumpy())


def run_model(
    itmax: int,
    m: int = M,
    n: int = N,
    *,
    backend: Any,
    device: core_defs.Device | None = None,
) -> tuple[jax.Array, jax.Array, jax.Array]:
    """Integrate `itmax` steps and return the final `u`, `v`, `p` as NumPy arrays."""
    model = Model(m, n, backend=backend, device=device)
    model.advance(itmax)
    return model.arrays()

## Checking the linear stages

The channels are the eight operators of `swm.ipynb` in tap form, so the check
of the linear stages is against shifts written with `numpy.roll`, on random
fields.

In [ ]:
def check_linear_stages() -> None:
    """Compare every channel of both stages against roll-based references."""
    rng = np.random.default_rng(42)
    m, n = 8, 10
    fields = {
        "u": rng.standard_normal((m, n)),
        "v": rng.standard_normal((m, n)),
        "p": rng.random((m, n)) + 2.0,
    }
    fields["cu"] = fields["cv"] = fields["z"] = fields["h"] = rng.standard_normal((m, n))

    def roll_avg_x(f):
        return 0.5 * (np.roll(f, -1, axis=0) + f)

    def roll_avg_y(f):
        return 0.5 * (np.roll(f, -1, axis=1) + f)

    def roll_avg_x_stag(f):
        return 0.5 * (np.roll(f, 1, axis=0) + f)

    def roll_avg_y_stag(f):
        return 0.5 * (np.roll(f, 1, axis=1) + f)

    stage1_ref = {
        "avgx_p": lambda: roll_avg_x(fields["p"]),
        "avgy_p": lambda: roll_avg_y(fields["p"]),
        "avgxy_p": lambda: roll_avg_x(roll_avg_y(fields["p"])),
        "dx_v": lambda: (np.roll(fields["v"], -1, axis=0) - fields["v"]) / DX,
        "dy_u": lambda: (np.roll(fields["u"], -1, axis=1) - fields["u"]) / DY,
        "avgxs_u2": lambda: roll_avg_x_stag(fields["u"] ** 2),
        "avgys_v2": lambda: roll_avg_y_stag(fields["v"] ** 2),
    }

    jax_fields = {name: jnp.asarray(a, F32) for name, a in fields.items()}
    stage1 = jax.device_get(linear_stage_1(jax_fields["u"], jax_fields["v"], jax_fields["p"]))
    for name, array in zip(OUT1, stage1):
        np.testing.assert_allclose(array, stage1_ref[name](), atol=1e-4, err_msg=name)

    stage2_ref = {
        "avgys_z": lambda: roll_avg_y_stag(fields["z"]),
        "avgxs_z": lambda: roll_avg_x_stag(fields["z"]),
        "avgys_avgx_cv": lambda: roll_avg_y_stag(roll_avg_x(fields["cv"])),
        "avgxs_avgy_cu": lambda: roll_avg_x_stag(roll_avg_y(fields["cu"])),
        "dx_h": lambda: (np.roll(fields["h"], -1, axis=0) - fields["h"]) / DX,
        "dy_h": lambda: (np.roll(fields["h"], -1, axis=1) - fields["h"]) / DY,
        "dxs_cu": lambda: (fields["cu"] - np.roll(fields["cu"], 1, axis=0)) / DX,
        "dys_cv": lambda: (fields["cv"] - np.roll(fields["cv"], 1, axis=1)) / DY,
    }
    stage2 = jax.device_get(
        linear_stage_2(jax_fields["z"], jax_fields["cu"], jax_fields["cv"], jax_fields["h"])
    )
    for name, array in zip(OUT2, stage2):
        np.testing.assert_allclose(array, stage2_ref[name](), atol=1e-4, err_msg=name)


check_linear_stages()
print("both linear stages agree with the shift-based references")

## Validation against NumPy

Agreement is checked against the NumPy reference above, which computes the same
scheme in float64. All differences below therefore include float32 roundoff of the
reference itself: the leapfrog scheme amplifies the f32 representation error
over the integration. The tolerances are absolute, as in `swm.ipynb`; they are
about an order of magnitude above the differences measured on the CPU, where
these cells were validated. A float32 trace of the reference `timestep`
shows the same drift, which is what pins it to precision rather than to the
reformulation.

In [ ]:
STEPS = 100

ATOL_UV = 2e-3  # peak |u|, |v| ~ 4; measured ~ 1e-4 after 100 steps
ATOL_P = 1.0  # peak |p| ~ 5e4; measured ~ 3e-2 after 100 steps, ~ 0.22 after 4000


def test_matches_numpy(m: int = M, n: int = N) -> None:
    """The model against `swm_numpy` after `STEPS` steps."""
    u, v, p = run_model(STEPS, m, n, backend=backend)
    u_ref, v_ref, p_ref = run(STEPS, m, n)
    print(
        "max differences:",
        f"du={np.max(np.abs(u - u_ref)):.3e}",
        f"dv={np.max(np.abs(v - v_ref)):.3e}",
        f"dp={np.max(np.abs(p - p_ref)):.3e}",
    )
    np.testing.assert_allclose(u, u_ref, atol=ATOL_UV)
    np.testing.assert_allclose(v, v_ref, atol=ATOL_UV)
    np.testing.assert_allclose(p, p_ref, atol=ATOL_P)


test_matches_numpy()
print("the MXU formulation and NumPy agree after", STEPS, "steps")

To separate formulation error from float32 roundoff, the same comparison
against `swm_numpy` computed in float32: the differences should be of the
same order as above, and small in absolute terms.

In [ ]:
def test_matches_f32_numpy(steps: int = STEPS) -> None:
    """The model against a float32 NumPy trace of the same scheme."""
    u, v, p = (a.astype(np.float32) for a in initial_conditions(M, N, DX, DY))
    uold, vold, pold = u.copy(), v.copy(), p.copy()
    for cycle in range(steps):
        tdt = DT if cycle == 0 else 2.0 * DT
        alpha = 0.0 if cycle == 0 else ALPHA
        unew, vnew, pnew, uold, vold, pold = timestep(u, v, p, uold, vold, pold, DX, DY, tdt, alpha)
        u, v, p = unew, vnew, pnew

    um, vm, pm = run_model(steps, backend=backend)
    print(
        "max differences:",
        f"du={np.max(np.abs(um - u)):.3e}",
        f"dv={np.max(np.abs(vm - v)):.3e}",
        f"dp={np.max(np.abs(pm - p)):.3e}",
    )
    np.testing.assert_allclose(um, u, atol=1e-3)
    np.testing.assert_allclose(vm, v, atol=1e-3)
    np.testing.assert_allclose(pm, p, atol=0.5)


test_matches_f32_numpy()
print("drift against the float32 trace is of roundoff size, not formulation")

## Validation against the NCAR reference

`u`, `v`, `p` after the full 4000-step benchmark of NCAR/SWM `ref/16x16`, the
data of `swm_reference.npz`, inlined below in base64 so that this cell too runs
without the repository. In float32 the accumulated representation error after
4000 leapfrog steps is roughly `dp ~ 0.2` on the CPU; the tolerance in this
cell is four times that.

In [ ]:
import base64
import io

# the data of swm_reference.npz, inlined so this cell runs without the repository
SWB_REFERENCE_NPZ = """
UEsDBC0AAAAIAAAAIQC3i14k//////////8FABQAdS5ucHkBABAAiAkAAAAAAAB+BgAAAAAAAM3W+z9UeRzH8ZnR
5LLTZNtKHrro8ogkoYtrvqd0kdwKqcilZmjdF5GY8ii3SmwpqZRuHhJKost2+R7dLWqbohExahdNpEzbDFPas+dz
xh+wP63fvvM9j+/7p+frIdfN29VjPZsVz0oyEghjN8cY2Roa2QdbG5kaGgVHxcTFBEUGRMUIhP/+vjwoPFZI/R67
JShaSJ2NLaxMDS2sZpkaigz/459Ogc8gqjreiX+3nVuR7tmBLdKavNZIxHhoML7i/rlQvLZxyoMn4WJUKq97X27Q
gTTu7S/MCupExcLXU9LsupHINEKkCOxEs0WlCZbUve6HlTfXUN9Pu9DGDlCGYstn31TRzWKclbi9wMarA5tPnc9b
XNCJX2SNn1Ya343V+zP2fkUD2ixy78CyZfrZSpwhLRHGHpXhlML7YY1LN+IaSeWdpu0ytFKUVVaYrETCNzfWclgs
Ipet8algNZsYNeXOV+H376he1a8aS93LXStf2lDfd/fPPF7CD8B+7F69vmMyPFO0sIGdo8SRLi32Xjos0mGq7I2d
H5tU73dEvyvmDGqSttlPfxm5nkN6fVZ5dDXL8Ue560UNsRMyKfZ4GNUuR3XmuderN3CI6qkxXdGDmsStA0667XJt
IrUlO0eLOptOSs/wou5nF+us+tgmR1JxXa1uriMaent3reSVHJvyDePyqPfLfjYhtlJ7Dqj5gMln7eH9+Vc/uhx2
5JFzXRfompziktIlYaM3b1Jg6e31ZoKbIajlSaaiJ1GB4uL0dsWWcgmHBRNLbdx4RHTuE5OzZ/iETV6t/WNXHlEf
sD2ORd0n96UnOlLf76lrWDXBLRgVvw/SsNuswCan+/0uUe/3KhdZaS/lkfEhgVEJ+fzhfXuFZ1PWYh6Z47VGK/Mk
lxxRoOfDDVTgtsVWeZa3w5AtV1DunqxA+b7lS/TLuYSf/eHUQA8e0Zx+VqJRxCfixz7u7nHnEQfs9EMOlXGJ33xd
121LUqAqu1n91e6hiHPU4PQl6r0S0fRjKur9WQOjC4eovYEL5iKnPP7wvvUe61VOf2uSXpsG3y7w5pBdOvim5ws5
zr/rXCG8FYACVm394vGXHDnXnuM3+HOIwRdt0+q+axLx2vF2jwa0CWnSuRPrqHMkwTdeRN2/jpPyAv+Uo22mr6uN
J25EGXMuXX5OvdeX9dz3CPV+UrdIXEHtnTFecnVMn/bwfp1/ht95Lov01phTzM5Q4uxkzqvGbBmuWKnKu6RyxC67
ciaNy5ChzV93eLqlKdFqbsl5kssiePeGWj74sInufT0JQupcOdbSQZiqRM1S39P5aTIklbRNVuWswE8FoT4xOTI8
pOk3Mot6P0S8I19B7aVXc8oP+7CH932IUeb+RzrxvJRly5e6d2BnrUf1fX+IsU3rhvsLzwfjBuPgWxu3iVFNpsHJ
uBkdSDIpIlcS0YnMalSbxq/sRpdnmbs7U2f2QHklnt6Baj8lpNoniJG1QqqbpwrGX64fO7flmRi/+l5UeZt6X2RX
0/+J2ptta4T1Y7qxer9qBvh3SAb/zu/A/+OuDtp/ugT8H2L8p84G/xv2g//wFeA/FUF/7uZAfx5KBHR/GnqhP9e4
9XR/POugPzfWQ3/U+0Fm4H+rlpz2H+EP/j0iDGj/11Q9tP++3eB/nBv4L5sM/qMywX9OL/Tn7UvoD5n0gO7Pgnbo
z4wU6I8G05/x2tAf9f7YNvDfPgX8pxSBf0K/nvbv3gr+d+qAf+8Q8H9qCfjXY/yv+BX6M+EL9Mf8iQ/dn6NMf6qY
/uhNh/6EMf1R79vkgv8DjP9iJfj3XtZM+9eRgn9fxr/xIPjfLQD/ftXg388J+iNwgP4sLAmg+9PG9OdHpj++86E/
Ew4y/WH2H/wE/ufZgv/UdeBfajGX9l/F+L/C+LeTg/8xZeDfTAX+ozZAfyYXQn+W69vT/XnGh/6c8Yf+hNtCfw5m
Qn/U+0WJ4J+zGPxfiwX/upwW2v8JE/DvnAn+W9+B/2bGf0oG+HcJgv6QP0B/1u10oPuj6wj9cX8A/UkUQ3/0taA/
6v0R2eB/C+P/cSP4Z12R0f4PhoP/Yj74lzSC/5GM/9aN4D+QBf05Ggf98fDspfvjvQ/6Y8b0J1YL+jOJ6Y96f5Dx
P+4j+L/YCv5dItto/xwS/KcowX+vBfj/xvivcQT/RYnQn+X60B9rCwHdn5km0J9pu6A/zXrQn6ar0B/1/v/l/49/
AFBLAwQtAAAACAAAACEApEOKM///////////BQAUAHYubnB5AQAQAIgJAAAAAAAAkgYAAAAAAADN1vtbTHkcwPGZ
yXR7xoR1a6OG1nZxSwkp+z2JpGhaRaXSzRSFMqUbYW032yS3Lq5JUaQtkWeRvodFuYVJmbLYaTMyenSZbWdqqD3O
Z+Yf2J92fpv55fOc8/m83s8c8Vy72sufyUhg7LTYKIgLF1os5lk4RSyymMWziIgRxgtDtwXHCDcKvv7uGrolTkD9
HrcpNFZAfbect3AWb95Cq1m8VN5//BjqzN2aqgyRoX029Z8FIyPoVMuBXP0hPeLN0QdOjas5xMVJjV3dfA6RklR6
ym9Ej8jJ7k4UsBmEnrUN332rDG3wHUJXT8hwWOZnNGjAIF/FfihjDemRFTW9q/JcOGS5yrtV5Mwhs3MWebgN6JH5
QZmB5WwGyXAYbbOhQIa181+nVSTamkiRJVuhHr9LhUaZZWT6rGcR6wNT4hkVbKLN0Tjy6CU24UFwLZdsYBGZY21/
EKSp0JCssgabS1HnyrnVGd5SXPtx+XLjAyo8LHq6XdefRVa62o+xPsMmbdet0c86zSb5gqG/7NeyyOrhWWXMTBX2
SF/uuowvxdr5HX0rb67ZIkZFATUvHVLkyPmsoUfvGwXi92ckuyQrUU/Qar+knUqUK/yTE9KpQCkvA4oL0+Uo+XVi
mlOiGEXnt/qskYixbucFQdwxObb7oPZ636bAlUuijcLDlLjt1CRfdogST+fim94vFPhCMqu95YAcN4xteNzzTIy1
84+q5cxgVRQmm81PXOAGY95e6YMxR1zQ/sYWj8meESiesOuv5Ueh8ofttZZTghDDdchUnbsCZzW3jclXR2Dh/T3V
d0ujcOOpuuiWZUG4+onvrzpiN7SKFTZn481IVNfkmG97KxrZiNZXC+qCEWk+kl+ldsFdCwV3F5RHYO1843tf1LFt
Ytw33D2p57gcV3fcWSdpV+CfPobqOIYrseSYSXEV9TzM76suN1PPg8KifIW5cvyt9HjppudifLbV7F4T9TxWXTX1
rdT7zD7vdT/mrQJ5P8tSdlPv0270xkr+LiXydN/xj9c7BfolK3fqhEw54htF1AUliZF2/tqElJMOPlIcvXvBE2au
Cp/h8uLzqf0+Lu0PrKL2W7rT/Lia2u+H/c0BBdR+rfQDdUXUfm/rn6+5Re03e+TRx0pqvyd2iy4VUfdFzDvyWy11
X/7xk/bFUfclCqhcalzJJuIelHKfUPc1jbXH2zNdhTakmpyO/06KtPNn8OZznE/K8JfVr5x8DBlk0yZrYgd17zHq
JQsNlnHIDpVR0TB1772yVHE1de/F4t2FSureV9je7u+j7t2pPqdIFCpDHp3X17EYDGKCufB9LOXNff6UCgdPDiF1
yEsL8eIQL1reTH9EeVuqf6GcpLyFGW09IqG8aed/ypk4vSKhCxebyTscA5lk9g9th6z/NiBLNoXEJBZyyRkXbVLd
8rnkTIul18b1GJDvrrIq83yZpIODBTYWdmFD/9dm6Y5dyFlfp+/kj0zC7JDbmLcKAwIdaLIuOcslJmSXSHTOcYnT
egmODYMGRMbt4VeffJlE7g112MSVXUg7v8QT/C/IAv/6MeC/7gr4t1WDf9cM8D8/BPwb24H/Cmvw/9wK+jMogf4w
C6A/d8dDf6o0/TEXQX8UE6E/2vleBPgf2wv+s3LA/xZ38I/9wP/vYeD/IGM27f/LNvAfngr+23WhP7Wm0B8/TX/q
F0N/IjX96eBCf8YpoD/a+QNR4D+wG/xH9YH/DBfw73UC/E80BP9r94H/EY1/lY477X+2D/Tncwn0Z8EA9MckAPrD
1vRHpunP3vfQH+38FPvttP/JObW0f8bLcNr/5Dl82v+1d6to/5Z5TrT/kED7gq/+eZt9aP+jysC/5M43dH9i7Rrp
/vxRJ6H7c7zZgu7PweBWuj+99V/o/jR5vKT7o52vIwX/yRr/T8vAf/Fj8M8dB/5rXMD/tsPgP2pQQPu/8Qr8P9f0
x+Q19KeoE/rjVgj9Ia2hP1MToT+m9dAf7fx7jMe0f/s08E/agP9rQvB/Phj8b9b4H6XeTfs/lwD+DyeD/4Y90J+f
R0N//M5Cf6JSoT9mIuhPkhH0R95/ju6Pdv7lBvCv0Ph/Zwb+H2r8S5zAv+cL8F80CvxHTgb/HtPAv7En9Mc0Evrj
PwT9mTkI/SE+Qn8GNP0hraA/2vmbt4L/dF3wn5cD/nUOg/84Efi/og/+C+rBf/p18C85BP5XTIX+7CGgPzWR0J+R
MujPltnQH8Na6I+ZPfRHO///8v/jX1BLAwQtAAAACAAAACEAglTpX///////////BQAUAHAubnB5AQAQAIgJAAAA
AAAATQQAAAAAAADNln9M1HUcxiH5EWRQcBIzhIPyfu0QXeUxsLhIgeM4d63S6B93yTkiEHbHcAb9sFIT4bSdkv3h
QB3ayKxMGaSAm5oJsqWBOFaaf1B3UtPY1c2zWO8br+9i/Nlf3T/P7nPf597PnvfzfO68q1+y2csjIxoiGrUVTvd6
lzZPrV2xIVdrVGs31LrqXY6N62pdFc7weZGj2u2Uc3elo84p73XLTEb1MpPeqH5T/R9f8fOrL9dZq3zm0e+D42Ov
+czDOZNpXsETQcPSTsG9upuH5snnQ9/odmwSzHkvK7lPMGHx6eRPwwj/dqL6xqg8/+yF4xt3CWZtvXnygKDxp/dv
R8rnixxf28P8tlN5t04LRq+y13UJKvNjH5rhW6oHm9cJ6rR7ClIFpzYHe/WCm7ot85oF89qjNOHvO7wk9kOL4Ejh
QGdueD76bfAbvl2bmSIYVdL3mUZwkavXul1woWrBrWlBq+eGvjg8P3T596dmzX8F/ZXw1/YsTLxU6TM/OVU/+oPg
yxMZ956Rc1cgqaZXsHQq6WyU8PerWgIBeX8V//ToL4U/Xhq4Oy5Yb/48lC/nB9/eMnxS8OFDTV9GCH+v+uyJO7Pm
Z+NfDPqvwi/6u+S5ScHiVb8+vVrOO/wPZg0KfrdvxYEU+Z7hjCvmWMHz7G8S/3LQb4Lf2Bb/apmcP3HquuqCYM/1
VE1y2Mdf+rvD+1bmp7O/HPyrQX8r/JGRLP8xwXdCW77Sy3OPlPf76wWn98UHKgS3kp8m9leOf/non4Bv1z7QpJPn
bLsbPnILdugyB9dX/Ts/nfwY2N8A/pWg3wn//iPZzR2CBQVtYz7BnzPavWNhX8ivjfw0sr8u/HsL/X8dnuF3pkzH
+QVD0Z7l12bN30Z+XyA/C9jfOfyzon8n/D0XQ54lr/vMSflnotIEdfSnh/wWk59R9heDf0vR3wLfEV3zSZivzA+t
nOmPnvweJT8B9teKf0H0u+F3Wnra4wRj6O8V+vOFkl/yc5f9/Yl/BvQfg6/Mj6O/cfTHRH7TyU8G+0vDvx3o/wN+
NPwp+jtOf3rJ7y7yc4b9ZeCfCv3K/En4ufTfRX+V/mjIbyr5KWJ/Mfg3H/1W+Nn010d/Gsmvmfx0s79B/FPmn0O/
Af5K+j9Ef6voTz/5VZEfD/uz4V8L+gvhJ9FfpT9vkF8T+fGyP2X+cfzzo98AX7k/PPTXQn8GyO8E+VH2F4l/59G/
Bv6L9Pdj+vMj+R0iP8p8D/urxb8y9JvhX6L/FfT3efqzjfw+Tn6Wsz83/m1Hv3J/bJ5z/wTJr2fu7xf7a8I/5f5s
hZ/A/fEB/b1DfxaTXw356WJ/Rvxbg/4U+Ep/ffRHma+Z0/9C9ncR/9ToN8BX7o8a+ruT/pSR30fJTx/7s+OfF/27
4T9Gf5X5ifRngPweIT/X2N89/AuhP3NO/yP5/U6gP63k9z7fTH7eZX+/4Z8T/UfhK/P/L/8//gFQSwECLQMtAAAA
CAAAACEAt4teJH4GAACICQAABQAAAAAAAAAAAAAAgAEAAAAAdS5ucHlQSwECLQMtAAAACAAAACEApEOKM5IGAACI
CQAABQAAAAAAAAAAAAAAgAG1BgAAdi5ucHlQSwECLQMtAAAACAAAACEAglTpX00EAACICQAABQAAAAAAAAAAAAAA
gAF+DQAAcC5ucHlQSwUGAAAAAAMAAwCZAAAAAhIAAAAA
"""


def test_matches_reference() -> None:
    """The model against the NCAR reference after all 4000 steps."""
    u, v, p = run_model(ITMAX, backend=backend)
    u_out, v_out, p_out = to_reference_layout(u, v, p)
    reference = np.load(io.BytesIO(base64.b64decode(SWB_REFERENCE_NPZ)))
    print(
        "max differences:",
        f"du={np.max(np.abs(u_out - reference['u'])):.3e}",
        f"dv={np.max(np.abs(v_out - reference['v'])):.3e}",
        f"dp={np.max(np.abs(p_out - reference['p'])):.3e}",
    )
    np.testing.assert_allclose(u_out, reference["u"], atol=ATOL_UV)
    np.testing.assert_allclose(v_out, reference["v"], atol=ATOL_UV)
    np.testing.assert_allclose(p_out, reference["p"], atol=ATOL_P)


test_matches_reference()
print("the MXU formulation reproduces the NCAR reference after", ITMAX, "steps in float32")

## The flow

`u`, `v` and `p` after `STEPS` steps.

In [ ]:
import matplotlib.pyplot as plt

u, v, p = run_model(STEPS, backend=backend)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, field, name in zip(axes, (u, v, p), ("u", "v", "p")):
    image = ax.contourf(field.T, levels=20)
    ax.set_title(f"{name} after {STEPS} steps")
    ax.set_aspect("equal")
    fig.colorbar(image, ax=ax)
fig.tight_layout()

## Performance: convolutions or explicit shifts

The baseline is the same scheme the `swm.ipynb` formulation compiles to on any
backend: each difference and average as an explicit shifted array, here
`numpy.roll` under `jax.jit`, in float32 like the model. Everything a TPU
executes as elementwise arithmetic on shifted views is VPU work, which is why
this shape of stencil is bandwidth-bound; the convolutions instead hand the
stencil to the MXU as one implicit-im2col GEMM per stage, reading every field
once per step and producing all shifts as channels.

Each program runs `WARMUP_STEPS` first (compilation included), then `REPEATS`
blocks of `TIMED_STEPS` steps between two waits for the device; the median
time per step over the blocks is reported. The comparison is opt-in. Start
the kernel with `PYTHONOPTIMIZE=1` to time without GT4Py's runtime checks, and
use a TPU runtime for the interesting half of the table -- on CPU and GPU the
convolutions can lose to the roll baseline.

In [ ]:
import statistics
import time

RUN_PERFORMANCE_COMPARISON = False

GRID_SIZES = (128, 256, 512, 1024, 2048, 4096)
WARMUP_STEPS = 5
TIMED_STEPS = 20
REPEATS = 5


@jax.jit
def roll_step(u, v, p, uold, vold, pold, tdt, alpha):
    """One timestep as explicit shifts: the `swm_numpy` scheme under `jax.jit`."""

    def avg_x(f):
        return 0.5 * (jnp.roll(f, -1, axis=0) + f)

    def avg_y(f):
        return 0.5 * (jnp.roll(f, -1, axis=1) + f)

    def avg_x_stag(f):
        return 0.5 * (jnp.roll(f, 1, axis=0) + f)

    def avg_y_stag(f):
        return 0.5 * (jnp.roll(f, 1, axis=1) + f)

    def delta_x(f):
        return (jnp.roll(f, -1, axis=0) - f) / DX

    def delta_y(f):
        return (jnp.roll(f, -1, axis=1) - f) / DY

    cu = avg_x(p) * u
    cv = avg_y(p) * v
    z = (delta_x(v) - delta_y(u)) / avg_x(avg_y(p))
    h = p + 0.5 * (avg_x_stag(u * u) + avg_y_stag(v * v))

    unew = uold + avg_y_stag(z) * avg_y_stag(avg_x(cv)) * tdt - delta_x(h) * tdt
    vnew = vold - avg_x_stag(z) * avg_x_stag(avg_y(cu)) * tdt - delta_y(h) * tdt
    pnew = (
        pold - (cu - jnp.roll(cu, 1, axis=0)) / DX * tdt - (cv - jnp.roll(cv, 1, axis=1)) / DY * tdt
    )

    uold = u + alpha * ((unew - u) + (uold - u))
    vold = v + alpha * ((vnew - v) + (vold - v))
    pold = p + alpha * ((pnew - p) + (pold - p))
    return unew, vnew, pnew, uold, vold, pold


class RollModel:
    """The same integration loop as `Model`, driving `roll_step`."""

    def __init__(self, m: int, n: int) -> None:
        self.u, self.v, self.p = initial_conditions(m, n, DX, DY)
        self.u, self.v, self.p = (jnp.asarray(a, F32) for a in (self.u, self.v, self.p))
        self.uold = jnp.array(self.u)
        self.vold = jnp.array(self.v)
        self.pold = jnp.array(self.p)
        self.cycle = 0

    def advance(self, cycles: int) -> jax.Array:
        for _ in range(cycles):
            tdt = DT if self.cycle == 0 else 2.0 * DT
            alpha = 0.0 if self.cycle == 0 else ALPHA
            self.u, self.v, self.p, self.uold, self.vold, self.pold = roll_step(
                self.u, self.v, self.p, self.uold, self.vold, self.pold, tdt, alpha
            )
            self.cycle += 1
        return self.p


def ns_per_step(runner: Any, steps: int) -> float:
    """Time `steps` iterations, waiting for the device before and after."""
    token = runner(steps)
    jax.block_until_ready(token)
    start = time.perf_counter()
    token = runner(steps)
    jax.block_until_ready(token)
    return (time.perf_counter() - start) / steps


if RUN_PERFORMANCE_COMPARISON:
    print(f"running on: {jax.devices()[0]}")
    print(f"{'grid':>9} | {'roll us/step':>12} | {'MXU us/step':>11} | {'speedup':>7}")
    print("-" * 50)
    for size in GRID_SIZES:
        roll_model = RollModel(size, size)
        roll_model.advance(WARMUP_STEPS)
        times = [
            ns_per_step(lambda steps: roll_model.advance(steps), TIMED_STEPS)
            for _ in range(REPEATS)
        ]
        roll_us = 1e6 * statistics.median(times)

        model = Model(size, size, backend=backend)
        model.advance(WARMUP_STEPS)
        times = [
            ns_per_step(lambda steps: model.advance(steps), TIMED_STEPS) for _ in range(REPEATS)
        ]
        mxu_us = 1e6 * statistics.median(times)

        print(f"{size:>9} | {roll_us:>12.1f} | {mxu_us:>11.1f} | {roll_us / mxu_us:>6.2f}x")

## Notes

- **What lands on the MXU here**: two convolutions per step, one per linear
  stage. Everything pointwise (the products of `kernel_terms` and the filter)
  runs on the VPU, which is fine: those terms read each channel once.
- **Precision**: the model is float32 because the MXU multiplies in `bfloat16`
  inputs with float32 accumulation and has no float64 at all. The validation
  section quantifies the cost of that with the 4000-step NCAR reference.
  `jax.default_matmul_precision` is the knob for the truncation used in
  float32 contractions.
- **Grid sizes**: the GEMMs pad `K` and `N` to the systolic array's tiles, so
  grids that are multiples of 128 (256 on newer TPUs) waste the least; the
  sweep above uses such sizes.
- **Multi-chip**: the domain decomposes along the two axes of a TPU slice
  over the ICI torus (`jax.sharding`), with the linear stages resharding the
  halo as all-gather style collectives. Not explored here.
- **Pallas**: writing the stages with `jax.experimental.pallas` would keep the
  fields resident in VMEM across the whole step, trimming the last HBM
  round trips of this formulation.